# Autoencoders: Learning to Compress, Clean, and Recommend

## What We Will Cover Today

- The core intuition behind autoencoders: how a network can learn to compress information and rebuild it
- Where autoencoders are used in practice: compression, denoising, and generating embeddings
- Building a simple autoencoder and understanding what the bottleneck layer represents mathematically
- Using autoencoder embeddings for feature extraction and transfer learning
- Denoising autoencoders: why adding noise on purpose makes a network more useful
- Recommender systems built on autoencoder embeddings
- How the same core idea behind autoencoders shows up in the AI systems you already use every day

## A Running Example: Nightwatch

Nightwatch is a citizen-science platform. Amateur astronomers point ordinary cameras and small telescopes at the sky and upload their photographs, and the Nightwatch team helps them catalogue what they have captured.

Maya is a data scientist on the Nightwatch team. Three problems have landed on her desk this week:

- The photo library has grown far faster than the storage budget, and every image is taking up more space than it needs to.
- A large fraction of the uploads are noisy: streetlights, sensor grain, and compression artifacts make the actual sky content hard to see.
- Users keep asking for a way to find "photos like this one" so they can discover other captures of the same region of sky.

Maya suspects a single family of neural networks can help with all three. We are going to work through her reasoning today, starting from the simplest possible version of the idea.

## Building the Intuition

Suppose Maya builds a small network where the input layer and the output layer have exactly the same number of neurons. She feeds in one image at a time, represented as a vector, and asks the network to produce that same vector back out.

Let us define this carefully, since we will reuse this notation all day:

- $x_i$ is the input vector for the $i$-th image (its pixel values, flattened into a list of numbers).
- $\hat{x}_i$ is the network's output for that same image: its attempt at reconstructing $x_i$.

**Question: can we build a network where $x_i$ and $\hat{x}_i$ end up identical?**

If every activation in the network is linear, yes, trivially: whatever goes in comes back out unchanged, so this version teaches us nothing useful. The interesting version happens once we add a nonlinearity and, more importantly, once we force the information to pass through a narrower layer than the input.

Consider a network shaped like this:

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/243/520/original/autoencoder.png?1790753299" height=500>

Now we cannot expect $x_i = \hat{x}_i$ exactly, only $x_i \approx \hat{x}_i$. How close the two are depends on how we measure the gap, which depends on what kind of values we are reconstructing:

- If each value in $x_i$ is binary (0 or 1), we use **binary cross-entropy**.
- If we are reconstructing a one-hot category across several classes, we use **categorical cross-entropy**.
- If the values are continuous, such as raw pixel brightness, we use **mean squared error (MSE)** or its square root, **RMSE**.

For continuous data, the reconstruction loss for one image looks like this:

$$L(x_i, \hat{x}_i) = \sum_{j=1}^{d} (x_{ij} - \hat{x}_{ij})^2$$

where $d$ is the number of features in one image (for example, the number of pixels), $x_{ij}$ is the value of the $j$-th feature of image $i$, and $\hat{x}_{ij}$ is the network's reconstruction of that same feature. Training simply looks for the weights that make this loss as small as possible, averaged across every image in the training set.

### Where the Real Value Is Hiding

Say the network narrows from $d$ dimensions down to a middle layer of only $k$ dimensions, where $k$ is much smaller than $d$, and then widens back out to $d$ on the way to the output.

If the network can still produce an output close to the original $d$-dimensional input, that tells us something important: **all of the information needed to rebuild the image was preserved in that narrow middle layer.** Nothing essential was thrown away between the input and that layer, only reshaped.

That narrow middle layer is called the **bottleneck**, and the vector of numbers sitting there for a given input is called its **latent representation**, or **embedding**. We can write this as:

- $z_i = f(x_i)$, where $f$ is the first half of the network (the **encoder**) and $z_i \in \mathbb{R}^k$ is the embedding of image $i$.
- $\hat{x}_i = g(z_i)$, where $g$ is the second half of the network (the **decoder**), turning the embedding back into a full-sized reconstruction.

```
x_i  →  [ Encoder f ]  →  z_i  (bottleneck, dimension k)  →  [ Decoder g ]  →  x̂_i
```

This whole structure, encoder plus bottleneck plus decoder trained to reconstruct its own input, is what we call an **Autoencoder**. Even though it looks like it is just learning to copy its input, the point of training one is almost never the copy itself. The point is $z_i$: a compressed, information-dense description of $x_i$ that we can reuse elsewhere.

### Quiz 1

****Nightwatch stores each photo as continuous brightness values per pixel, not as 0/1 values. Which loss function is the natural first choice for training an autoencoder on this data?****

a) Categorical cross-entropy, because images belong to categories such as "nebula" or "star cluster"

b) Binary cross-entropy, because pixels represent brightness values as either dark or bright

c) Mean squared error, because pixel brightness is continuous and we want to penalize reconstruction error

d) Accuracy, because we want to measure the fraction of pixels reconstructed perfectly

****Answer: (c)****

****Explanation:**** Pixel brightness in a photograph is continuous, not binary or categorical, so MSE (or RMSE) is the natural match. Binary cross-entropy is reserved for genuinely binary targets, categorical cross-entropy for one-hot class targets, and accuracy is a classification metric, not a reconstruction loss.


## Applications of Autoencoders

Now that we know what an autoencoder produces, a compressed embedding at the bottleneck, we can list the broad families of problems that embedding is useful for:

1. **Dimensionality reduction / compression** — fewer numbers to store or transmit per item, which means faster downstream computation and lower latency.
2. **Denoising** — training the network to map a corrupted version of the input back to the clean version.
3. **Embeddings for downstream use** — the bottleneck vector can be reused for recommendation, clustering, similarity search, and as input features to other models.

This maps directly onto Maya's three problems: compression addresses the storage budget, denoising addresses the noisy uploads, and embeddings address the "find photos like this one" request.

### Ask AI

After class, ask an AI assistant: *"Besides compression, denoising, and recommendation, what are three other real-world products or industries that rely on autoencoders or similar embedding-based networks, and what does the bottleneck represent in each case?"* Compare its examples with the ones we covered today.

## Dimensionality Reduction, in Depth

**Question: could we design a network that behaves like PCA?**

Say we have data in $d$ dimensions, and we want a representation in $d'$ dimensions, where $d' < d$. This is exactly the encoder half of an autoencoder: $d$-dimensional input in, $d'$-dimensional embedding out.

Here we do not actually need the decoder's output for the final task. We only need the embedding. The fact that a decoder *can* rebuild the original input from that embedding is what proves the embedding was information-rich enough to be trustworthy.

**Question: must the encoder and decoder be mirror images of each other?**

Not anymore. Early autoencoder designs often kept the encoder and decoder symmetric (same number of layers, same neuron counts in reverse) and sometimes even shared weights between them, a trick called **weight tying**, mainly to cut down the number of parameters to train. Today, with enough data and compute, there is no requirement for symmetry: the encoder and decoder can have entirely independent architectures.

### Quiz 2

****Which statement about encoder and decoder design is correct?****

a) The encoder and decoder must always have the same number of layers and neurons, or training will fail

b) Symmetry and weight tying were common early choices, mainly to reduce parameter count, but neither is required today.

c) Weight tying between encoder and decoder is required for every autoencoder model to train

d) The decoder must always be larger than the encoder, regardless of the task or architecture used

****Answer: (b)****

****Explanation:**** Symmetric, weight-tied designs were a historical convenience for keeping parameter counts down. Modern autoencoders are free to use independent, asymmetric encoder and decoder architectures, since the constraint was never mathematical, only practical.


### Interactive Playground: Bottleneck Size vs. Reconstruction Quality

Training a full neural autoencoder live during class takes too long to be a good demo. Instead, since we just established that the encoder is doing a PCA-like job, this playground uses PCA itself as a stand-in encoder-decoder pair: it projects each image down to $k$ components (the bottleneck) and back up, which mirrors exactly the compress-then-reconstruct behavior of the network we discussed above.

Move the slider to change $k$, the bottleneck size, and watch the reconstructed digit change. Use this to show, live, why a very small bottleneck loses detail and why reconstruction quality improves as $k$ grows, without needing to train anything.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from ipywidgets import interact, IntSlider

# We use scikit-learn's small built-in digit scans purely as a fast,
# easy-to-run stand-in for Nightwatch's telescope photos. The same
# compress-and-reconstruct idea applies regardless of what the images show.
digits = load_digits()
images = digits.data          # shape: (n_samples, 64), each row is an 8x8 image flattened
image_shape = (8, 8)
sample_index = 7               # a fixed image we will compress and reconstruct

def show_reconstruction(k):
    pca = PCA(n_components=k)
    embeddings = pca.fit_transform(images)          # the "bottleneck" vectors, shape (n_samples, k)
    reconstructed = pca.inverse_transform(embeddings)  # the "decoder" step, back to 64 dimensions

    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(images[sample_index].reshape(image_shape), cmap="gray")
    axes[0].set_title("Original (64 dims)")
    axes[0].axis("off")

    axes[1].imshow(reconstructed[sample_index].reshape(image_shape), cmap="gray")
    axes[1].set_title(f"Reconstructed (bottleneck k={k})")
    axes[1].axis("off")
    plt.show()

interact(show_reconstruction, k=IntSlider(min=1, max=64, step=1, value=8));

interactive(children=(IntSlider(value=8, description='k', max=64, min=1), Output()), _dom_classes=('widget-int…

### Code Walkthrough: A Real Autoencoder

Now let's build the actual neural version, encoder and decoder as trainable layers, rather than the PCA stand-in above. We will compress each 64-dimensional digit scan down to an 8-dimensional bottleneck.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import load_digits
import numpy as np

digits = load_digits()

X = digits.data.astype("float32") / 16.0   # pixel values here range 0-16, scale to 0-1

X_train, X_test = X[:1500], X[1500:]

input_dim = X.shape[1]      # 64, the flattened 8x8 image

bottleneck_dim = 8          # the compressed representation size


class Autoencoder(nn.Module):
    def __init__(self, input_dim, bottleneck_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), # (64, 32)
            nn.ReLU(),
            nn.Linear(32, bottleneck_dim), # (32, 8)
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 32), # (8, 32)
            nn.ReLU(),
            nn.Linear(32, input_dim), # (32, 64)
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


autoencoder = Autoencoder(input_dim, bottleneck_dim)

criterion = nn.MSELoss()

optimizer = optim.Adam(
    autoencoder.parameters()
)


X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)


epochs = 40
batch_size = 32

for epoch in range(epochs):

    autoencoder.train()

    # shuffle=True equivalent
    indices = torch.randperm(X_train_tensor.size(0))

    for start in range(0, X_train_tensor.size(0), batch_size):

        batch_indices = indices[start:start + batch_size]

        X_batch = X_train_tensor[batch_indices]

        optimizer.zero_grad()

        reconstructed = autoencoder(X_batch)

        loss = criterion(reconstructed, X_batch)

        loss.backward()

        optimizer.step()


# Validation
autoencoder.eval()

with torch.no_grad():
    reconstructed = autoencoder(X_test_tensor)
    validation_loss = criterion(reconstructed, X_test_tensor)
    print(validation_loss.item())
print("Final validation loss:", validation_loss.item())

0.03220740705728531
Final validation loss: 0.03220740705728531


In [ ]:
# The encoder alone gives us the embeddings we actually care about
autoencoder.eval()

with torch.no_grad():
    embeddings = autoencoder.encoder(X_test_tensor)

print("Original dimension:", X_test.shape[1])
print("Embedding dimension:", embeddings.shape[1])

Original dimension: 64
Embedding dimension: 8


## Why Use an Autoencoder Instead of PCA?

At first glance, **PCA and an autoencoder seem to do the same thing**:

- **PCA:** compresses data from a higher dimension to a lower dimension.
- **Autoencoder:** the **encoder** compresses data into a lower-dimensional representation, and the **decoder** reconstructs it.

So, why not just use PCA?

The key difference is **what kind of patterns they can learn**.

| PCA | Autoencoder |
|---|---|
| Learns **linear** transformations | Can learn **non-linear** transformations |
| Finds directions of maximum variance | Learns representations optimized for reconstruction |
| Uses a fixed mathematical procedure | Learns representations using a neural network |
| Works well when the data has mostly linear structure | Can capture complex, non-linear structure |
| Usually simpler and faster | More flexible, but more computationally expensive |

### The Main Idea

PCA essentially asks:

> **"What linear combination of the original features captures the most variance?"**

An autoencoder can ask something more flexible:

> **"What representation allows me to reconstruct the original data as accurately as possible, even when the underlying structure is non-linear?"**

For simple, approximately linear data, **PCA can be sufficient**. But for complex data such as images, text, or other highly non-linear data, an autoencoder can learn richer representations.

**In short:**

$$
\boxed{\text{PCA} \rightarrow \text{Linear dimensionality reduction}}
$$

$$
\boxed{\text{Autoencoder} \rightarrow \text{Potentially non-linear dimensionality reduction}}
$$

## Feature Extraction and Transfer Learning

### Feature Extraction

Once we have embeddings, we are free to leave the autoencoder behind and use those embeddings as input features to any other model.

**Question: could Maya feed raw pixel values straight into a decision tree or logistic regression to classify what kind of object is in an image?**

Not well. Models like decision trees and plain logistic regression do not handle raw, high-dimensional pixel data gracefully; they have no built-in notion of spatial structure. What works much better is to first pass the image through a trained encoder to get its embedding, and then hand that compact, information-dense vector to the simpler downstream model.

For Nightwatch, this means: train an autoencoder once on a large pool of unlabeled sky photos, then use the resulting embeddings to train a lightweight classifier (KNN, logistic regression, or a small tree-based model) on a much smaller labeled set, for example, a set of photos Maya's volunteers have manually tagged as "galaxy", "star cluster", or "meteor trail".

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch

labels = digits.target   # for this demo only, standing in for Nightwatch's volunteer-tagged labels

autoencoder.eval()
with torch.no_grad():
    # Convert X to a PyTorch tensor before passing to the encoder
    embeddings = autoencoder.encoder(torch.tensor(X, dtype=torch.float32)).numpy()

emb_train, emb_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.3, random_state=42
)

clf = LogisticRegression(max_iter=1000)
clf.fit(emb_train, y_train)
preds = clf.predict(emb_test)

print("Accuracy using embeddings as features:", accuracy_score(y_test, preds))

Accuracy using embeddings as features: 0.8462962962962963


### Transfer Learning

Now imagine a much larger, more expensive network: one trained for weeks on powerful hardware by a research team, originally built to classify medical scans as cancerous or not.

**What are Maya's options if she wants to detect a completely different pattern, say, a rare comet trail, in her sky photos?**

**Option 1: train a brand-new network from scratch.** This is possible, but it is unlikely to match the quality of a model that took a well-resourced team weeks of compute to tune.

**Option 2: reuse the pretrained network's embeddings.** Even though that network was never trained on astronomy photos, its intermediate layers learned general-purpose patterns (edges, textures, shapes). Maya can feed her sky photos into that existing network, take the output of an intermediate layer as an embedding, and train a small new model on top of those embeddings for her comet-trail task.

Reusing a network trained for one task as a feature extractor for a different, related task is called **Transfer Learning**.

### Quiz 3

****A team trains an autoencoder on general product photos for an online store. A different team later reuses that autoencoder's encoder to help classify a completely unrelated set of images: handwriting samples for a separate project. What is this second step an example of?****

a) Transfer learning, because a network trained for one purpose is reused for a different task

b) Denoising, because the encoder removes unwanted variations from the input images

c) Dimensionality reduction, because the encoder produces fewer features than the input

d) Weight tying, because the encoder and decoder share parameters during training

****Answer: (a)****

****Explanation:**** Reusing a model (or part of one) that was trained on one task to help with a different task is the definition of transfer learning. The fact that the two tasks (product photos vs. handwriting) are unrelated is exactly what makes this transfer learning rather than simply continuing to train on the original task.


### Ask AI

After class, ask an AI assistant: *"How does transfer learning show up in large language models, compared to the image-based transfer learning example from an autoencoders class? What gets reused, and what gets retrained?"* Bring one interesting difference you find back to the cohort chat.

## Denoising Autoencoders

Sometimes an encoder is built with *more* neurons than the input, not fewer, so information is being expanded rather than compressed on the way to the middle layer. In that situation, and even sometimes with a genuine bottleneck, an autoencoder can take a shortcut: it learns the **identity function**, simply copying input to output without learning anything meaningful about the structure of the data.

**How do we stop the network from taking that shortcut?**

We deliberately corrupt the input with random noise before it ever reaches the encoder, but we still ask the network to reconstruct the original, clean version as its target.

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/243/534/original/denoising_.png?1790756714" width=800>

Since the noise has no consistent pattern, the network cannot learn to reproduce it. The only way to get close to the clean target is to learn the actual underlying structure of the data and ignore the noise. This acts as a form of **regularization**: the corruption forces the network to generalize instead of memorizing.

In [ ]:
import numpy as np

X = digits.data.astype("float32") / 16.0
X_train, X_test = X[:1500], X[1500:]

noise_factor = 0.4
rng = np.random.default_rng(42)

X_train_noisy = X_train + noise_factor * rng.normal(loc=0.0, scale=1.0, size=X_train.shape)
X_test_noisy = X_test + noise_factor * rng.normal(loc=0.0, scale=1.0, size=X_test.shape)

X_train_noisy = np.clip(X_train_noisy, 0.0, 1.0)
X_test_noisy = np.clip(X_test_noisy, 0.0, 1.0)

print("Clean training shape:", X_train.shape)
print("Noisy training shape:", X_train_noisy.shape)

Clean training shape: (1500, 64)
Noisy training shape: (1500, 64)


In [ ]:
denoiser_input = torch.tensor(X_train_noisy, dtype=torch.float32)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)

X_test_noisy_tensor = torch.tensor(X_test_noisy, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)


class DenoisingAutoencoder(nn.Module):
    def __init__(self, input_dim, bottleneck_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, bottleneck_dim),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


denoising_autoencoder = DenoisingAutoencoder(input_dim, bottleneck_dim)

criterion = nn.MSELoss()
optimizer = optim.Adam(denoising_autoencoder.parameters())

# Note the target here is the CLEAN data, even though the input is noisy
epochs = 40
batch_size = 32

for epoch in range(epochs):

    denoising_autoencoder.train()

    # shuffle=True
    indices = torch.randperm(denoiser_input.size(0))

    for start in range(0, denoiser_input.size(0), batch_size):

        batch_indices = indices[start:start + batch_size]

        X_batch_noisy = denoiser_input[batch_indices]
        X_batch_clean = X_train_tensor[batch_indices]

        optimizer.zero_grad()

        denoised_output = denoising_autoencoder(X_batch_noisy)

        loss = criterion(denoised_output, X_batch_clean)

        loss.backward()

        optimizer.step()


denoising_autoencoder.eval()

with torch.no_grad():
    denoised_output = denoising_autoencoder(X_test_noisy_tensor)
    validation_loss = criterion(denoised_output, X_test_tensor)

print("Final validation loss:", validation_loss.item())

Final validation loss: 0.047541242092847824


### Quiz 4

****Why does training on noisy inputs with clean targets discourage the network from simply memorizing (the identity function)?****

a) Because adding noise makes each training sample smaller and easier to process

b) Because noisy inputs allow the network to train faster without learning the underlying data structure

c) Because adding noise increases the number of neurons available to reconstruct each input sample

d) Because random noise varies across samples, so the network must learn the underlying data structure to reduce reconstruction error

****Answer: (d)****

****Explanation:**** The network is scored against the clean target, not the noisy input. Since the specific noise added to each sample is random and non-repeating, there is no pattern in it for the network to exploit, so the only path to a lower loss is genuinely modeling the underlying signal.


## Recommender Systems Using Autoencoders

Recall the idea of a user-item interaction matrix: rows represent items, columns represent users, and each cell holds some signal of interaction (a rating, a "starred" flag, a view count). For Nightwatch, an item is a photo, a user is a platform member, and the signal could be whether that member starred that photo.

This matrix is almost always **sparse**: any one user has only interacted with a tiny fraction of all photos.

**The idea:** treat each item's row of interactions as an input vector, and train an autoencoder on those rows. The bottleneck embedding for each item captures a dense summary of "the kinds of users who engage with this item," which turns out to be a strong signal of similarity between items.

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/243/539/original/ChatGPT_Image_Sep_30__2026__02_02_30_PM.png?1790757238" width=800>

Once every item has an embedding, we can compare any two items with **cosine similarity**:

$$\text{sim}(z_a, z_b) = \frac{z_a \cdot z_b}{\lVert z_a \rVert \, \lVert z_b \rVert}$$

where $z_a$ and $z_b$ are the embedding vectors for items $a$ and $b$, $z_a \cdot z_b$ is their dot product, and $\lVert z_a \rVert$, $\lVert z_b \rVert$ are their vector lengths. A value close to 1 means the two items tend to attract very similar users; a value close to 0 means they do not.

In [ ]:
import numpy as np
import pandas as pd

# A synthetic stand-in for Nightwatch's user-photo interaction matrix.
# In production this would be built from real starring/viewing activity;
# here we generate a small sparse matrix so the class can run this live.
rng = np.random.default_rng(7)
n_items, n_users = 300, 120

interaction = rng.random((n_items, n_users))
mask = rng.random((n_items, n_users)) < 0.08   # only ~8% of entries are filled in, like real sparse data
interaction_matrix = np.where(mask, np.round(interaction * 5), 0).astype("float32")

sparsity = (interaction_matrix > 0).sum() / interaction_matrix.size
print("Matrix shape (items x users):", interaction_matrix.shape)
print(f"Fraction of filled entries: {sparsity:.2%}")

Matrix shape (items x users): (300, 120)
Fraction of filled entries: 7.16%


In [ ]:
item_input = torch.tensor(interaction_matrix, dtype=torch.float32)


class ItemAutoencoder(nn.Module):
    def __init__(self, n_users):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(n_users, 64),
            nn.ReLU(),
            nn.Linear(64, 16),
            nn.ReLU()
        )

        self.decoder = nn.Linear(16, n_users)

    def forward(self, x):
        h = self.encoder(x)
        decoded_row = self.decoder(h)
        return decoded_row


item_autoencoder = ItemAutoencoder(n_users)

criterion = nn.MSELoss()
optimizer = optim.Adam(item_autoencoder.parameters())

epochs = 60
batch_size = 32

for epoch in range(epochs):

    item_autoencoder.train()

    # shuffle=True
    indices = torch.randperm(item_input.size(0))

    for start in range(0, item_input.size(0), batch_size):

        batch_indices = indices[start:start + batch_size]

        interaction_batch = item_input[batch_indices]

        optimizer.zero_grad()

        decoded_row = item_autoencoder(interaction_batch)

        loss = criterion(decoded_row, interaction_batch)

        loss.backward()

        optimizer.step()


item_autoencoder.eval()

with torch.no_grad():
    item_embeddings = item_autoencoder.encoder(item_input)

print("Item embedding shape:", item_embeddings.shape)

Item embedding shape: torch.Size([300, 16])


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(item_embeddings)
similarity_df = pd.DataFrame(similarity_matrix)

query_item = 42   # pretend a user is currently viewing photo #42
top_matches = similarity_df[query_item].sort_values(ascending=False)[1:6]  # exclude itself

print(f"Photos most similar to item {query_item}:")
print(top_matches)

Photos most similar to item 42:
152    0.948086
205    0.942094
132    0.934484
190    0.931542
227    0.925916
Name: 42, dtype: float32


### Quiz 5

****Why do we train the autoencoder on rows of the sparse interaction matrix rather than directly comparing raw interaction rows with cosine similarity?****

a) Raw rows cannot be compared using cosine similarity because they contain mostly zero values

b) The dense embedding compresses each item's interaction pattern into a smaller vector that captures structure and makes similarity comparisons more efficient

c) Training an autoencoder removes the need for any similarity metric when comparing items with each other

d) The autoencoder converts the original interaction ratings into binary labels before computing item similarities

****Answer: (b)****

****Explanation:**** You technically could compute cosine similarity directly on the raw sparse rows, but with mostly-zero entries and high dimensionality (one dimension per user), the result is noisy and expensive to compute at scale. The embedding step gives a compact, denoised summary of each item's interaction pattern, which is what makes similarity search fast and reliable in practice.


### Ask AI

After class, ask an AI assistant: *"How does an autoencoder-based recommender compare to classic collaborative filtering approaches like matrix factorization? In what situations would a team prefer one over the other?"* Note one trade-off you did not expect.

## Where This Idea Shows Up in the AI Systems You Already Use

Everything we built today, mapping a piece of data into a dense vector that captures its meaning, is not unique to autoencoders. It is the same core idea behind the **embedding models** that sit underneath most modern AI products.

- When a search engine or an AI assistant needs to find documents "similar in meaning" to your query, it typically encodes both the query and the documents into dense vectors using an embedding model, then compares them with a similarity metric, exactly the same cosine similarity we used for Nightwatch's photos.
- **Vector databases**, which store and search millions of these embeddings efficiently, exist specifically to make this kind of similarity search fast at scale.
- **Retrieval-Augmented Generation (RAG)**, a pattern many AI assistants use to answer questions about a specific set of documents, works by embedding your question, retrieving the most similar stored embeddings, and feeding the retrieved text to a language model as context before it answers. The retrieval step is, structurally, the same nearest-neighbor-in-embedding-space idea from our recommender section.
- Multimodal AI systems that can compare an image and a piece of text (for example, "does this photo match this caption") rely on encoders trained so that semantically related items end up close together in the same embedding space, a direct generalization of the encoder half of what we built today.

The specific architectures have grown far larger and are trained on far more data than what we used in class, but the underlying move, an encoder that compresses meaningful structure into a dense vector, is the same move you now understand from first principles.

## Wrap-Up

- An autoencoder is an encoder and a decoder trained together to reconstruct their own input; the useful output is almost always the compressed bottleneck vector, not the reconstruction itself.
- The choice of reconstruction loss (MSE/RMSE, binary cross-entropy, categorical cross-entropy) depends on the type of data being reconstructed.
- Encoder and decoder architectures do not need to be symmetric or share weights; that was a historical convenience, not a requirement.
- Bottleneck embeddings can be reused as features for downstream models (feature extraction) or repurposed from a model trained on a different task altogether (transfer learning).
- Denoising autoencoders are trained on corrupted inputs with clean targets, which prevents the network from taking the identity-function shortcut and forces it to learn real structure.
- The same encoder idea, applied to a sparse user-item interaction matrix, produces item embeddings that power similarity-based recommendations.
- The exact same principle, compressing data into a meaning-preserving dense vector, underlies embedding models, vector databases, and retrieval-augmented generation in modern AI systems.